# H-005 · Reshuffle cadence

`freeze` vs `quarterly` vs `12m6m` (12-month formation / 6-month trading). Soft N=35.
Requires Pass B `s2_rebalance_1d.csv`. Freeze `BOOK_STAR`.

Do not open the sealed holdout in this notebook.

Next: run H-006.


## 0. Imports & Config


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.report import (
    arm_selection_table,
    fold_table,
    fold_val_metrics,
    full_is_metrics,
    load_star_stack,
    median_sharpe_hint,
    plot_fold_boxplots,
    require_star,
    save_star_stack,
    write_tearsheet_pdf,
)
from backtest.s2_coint.research import (
    config_from_stack,
    is_end_for_stack,
    load_s1_weekly,
    load_s2_cuts,
    load_universe_panels,
    lookbacks_for_bar,
    n_trials_local,
    n_trials_stack,
    overlay_ols_hedge,
    overlay_kalman_hedge,
    register_hypothesis_arms,
    s2_artifacts_dir,
    s2_data_dir,
    tearsheet_path,
)
from backtest.s2_coint.runner import run_s2_backtest
from backtest.s2_coint.walkforward import embargo_bars_for_config, make_s2_folds

VERSION = "v2"
STAR_PATH = os.path.join(s2_artifacts_dir(version=VERSION, root=ROOT), "s2_star_stack.json")
LEDGER_PATH = os.path.join(os.path.dirname(STAR_PATH), "s2_variant_ledger.json")
cuts = load_s2_cuts(root=ROOT, version=VERSION)
stack = load_star_stack(STAR_PATH)
print("yahoo_drop_rate", cuts.get("yahoo_drop_rate"))
print("stack keys", sorted(stack.keys()))

from backtest.s2_coint.research import rebalance_path
from backtest.s2_coint.rotation import simulate_frozen_book, simulate_rotating_book

HYP_ID = "H-005"
CADENCE_ARMS = ("freeze", "quarterly", "12m6m")
SOFT_N = int(stack.get("N_STAR") or 35)

require_star("POOL_STAR", stack.get("POOL_STAR"))
require_star("FDR_STAR", stack.get("FDR_STAR"))
PAIRS_STAR = list(stack.get("PAIRS_STAR") or [])
if not PAIRS_STAR:
    raise ValueError("PAIRS_STAR empty — freeze H-001/H-002 first")
BAR = str(stack.get("BAR_STAR") or "1d")
train, full = load_universe_panels(None, BAR, PAIRS_STAR, root=ROOT, version=VERSION)
lb0 = lookbacks_for_bar(BAR)
train = overlay_ols_hedge(train, ols_window=lb0["ols_window"], z_window=lb0["z_window"],
                          hl_window=lb0["hl_window"], adf_window=lb0["adf_window"])
full = overlay_ols_hedge(full, ols_window=lb0["ols_window"], z_window=lb0["z_window"],
                         hl_window=lb0["hl_window"], adf_window=lb0["adf_window"])
is_end = is_end_for_stack(stack, full if BAR == "1h" else train)
if BAR == "1d":
    is_panel, oos_panel = train.copy(), full.loc[pd.to_datetime(full["date"]) > is_end].copy()
else:
    from backtest.s2_coint.research import split_is_oos
    is_panel, oos_panel = split_is_oos(full, is_end=is_end)
s1_weekly = load_s1_weekly(ROOT)
print("bar", BAR, "is_end", is_end, "IS", len(is_panel), "OOS", len(oos_panel), "pairs", len(PAIRS_STAR))



## 1. Load Pass B schedules


In [ ]:
reb_path = rebalance_path(BAR, version=VERSION, root=ROOT)
if not os.path.isfile(reb_path):
    raise FileNotFoundError(
        f"missing {reb_path}. Re-run s2_pair_panel.ipynb with BOOK_SCOPE='freeze_plus_rotate'."
    )
reb = pd.read_csv(reb_path, parse_dates=["rebalance_date", "effective_date"])
print(reb.groupby(["schedule", "lookback"]).size())
# Soft L=252 schedules for the cadence bake-off (L screened in H-007).
q = reb.loc[(reb["schedule"] == "quarterly") & (reb["lookback"] == 252)].copy()
s = reb.loc[(reb["schedule"] == "12m6m") & (reb["lookback"] == 252)].copy()
print("quarterly rows", len(q), "12m6m rows", len(s))


## 2. Score cadence arms on research IS


In [ ]:
# Freeze arm: fixed PAIRS_STAR. Rotate arms: use schedule masks via rotating simulator when available.
# Fallback: fold_val on freeze book for freeze; for rotate, restrict to pairs active most of the fold.
register_hypothesis_arms(HYP_ID, list(CADENCE_ARMS), overwrite=True, path=LEDGER_PATH)
panel = is_panel.copy()
dates = pd.DatetimeIndex(pd.to_datetime(panel["date"])).sort_values().unique()
folds = make_s2_folds(dates, n_folds=3, embargo_bars=embargo_bars_for_config(bar=BAR))
display(fold_table(folds))
cfg0 = config_from_stack(stack)
configs = {arm: cfg0 for arm in CADENCE_ARMS}

def active_pairs(schedule_df: pd.DataFrame, start, end) -> list[str]:
    sub = schedule_df.loc[
        (schedule_df["effective_date"] <= end) & (schedule_df["rebalance_date"] >= start - pd.Timedelta(days=200))
    ]
    # Top-N by rank at last rebalance before end
    if sub.empty:
        return list(PAIRS_STAR)[:SOFT_N]
    last = sub.loc[sub["rebalance_date"] == sub["rebalance_date"].max()]
    last = last.sort_values("rank").head(SOFT_N)
    return [str(p) for p in last["pair_id"]]

fold_parts, full_parts = [], []
for arm in CADENCE_ARMS:
    if arm == "freeze":
        ids = list(PAIRS_STAR)[:SOFT_N]
    elif arm == "quarterly":
        ids = active_pairs(q, dates.min(), dates.max())
    else:
        ids = active_pairs(s, dates.min(), dates.max())
    p = panel.loc[panel["pair_id"].astype(str).isin(set(ids))].copy()
    fold_parts.append(fold_val_metrics(p, folds, {arm: configs[arm]}, hyp_id=HYP_ID, s1_weekly=s1_weekly))
    full_parts.append(full_is_metrics(p, {arm: configs[arm]}, hyp_id=HYP_ID, s1_weekly=s1_weekly))
fold_df = pd.concat(fold_parts, ignore_index=True)
full_is_df = pd.concat(full_parts, ignore_index=True)
plot_fold_boxplots(fold_df, title="H-005 cadence fold-val")
plt.show()
print("median-Sharpe hint (commentary only):", median_sharpe_hint(fold_df))
display(arm_selection_table(fold_df, full_is_df))


## 3. Type `BOOK_STAR`


In [ ]:
BOOK_STAR = ""  # type after the table: "freeze" | "quarterly" | "12m6m"
require_star("BOOK_STAR", BOOK_STAR)
stack["BOOK_STAR"] = BOOK_STAR
save_star_stack(STAR_PATH, stack)
print("wrote", STAR_PATH, "BOOK_STAR", BOOK_STAR)


## 4. Notes for next hyp

Next: run H-006.
